In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers accelerate peft bitsandbytes flask pyngrok sentencepiece
print("Dependencies installed")

In [ ]:
# Cell 2: Locate and prepare LoRA adapter
import os
import zipfile

print("Available datasets in /kaggle/input")
print("=" * 50)
for folder in os.listdir("/kaggle/input"):
    folder_path = f"/kaggle/input/{folder}"
    print(f"{folder}/")
    if os.path.isdir(folder_path):
        for f in os.listdir(folder_path)[:10]:
            fpath = os.path.join(folder_path, f)
            size = f" ({os.path.getsize(fpath)/1e6:.1f} MB)" if os.path.isfile(fpath) else ""
            print(f"  - {f}{size}")

LORA_EXTRACT_PATH = "/kaggle/working/llama_lora"

def find_and_extract_lora():
    """Find LoRA adapter folder or zip and return usable adapter path."""
    print("\nSearching for LoRA adapter...")

    # Already extracted adapter
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            print(f"Found adapter folder: {root}")
            return root

    # Candidate zip files
    zip_found = None
    keywords = ("lora", "llama", "adapter")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            f_lower = f.lower()
            if f.endswith(".zip") and any(k in f_lower for k in keywords):
                zip_found = os.path.join(root, f)
                print(f"Found adapter zip: {zip_found}")
                break
        if zip_found:
            break

    # Fallback to any zip if needed
    if not zip_found:
        for root, dirs, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".zip"):
                    zip_found = os.path.join(root, f)
                    print(f"Found generic zip: {zip_found}")
                    break
            if zip_found:
                break

    if not zip_found:
        print("No LoRA adapter found in /kaggle/input")
        return None

    os.makedirs(LORA_EXTRACT_PATH, exist_ok=True)
    print(f"\nExtracting adapter from: {zip_found}")
    with zipfile.ZipFile(zip_found, "r") as z:
        z.extractall(LORA_EXTRACT_PATH)

    # Check root first
    root_cfg = os.path.join(LORA_EXTRACT_PATH, "adapter_config.json")
    if os.path.exists(root_cfg):
        print(f"Adapter ready: {LORA_EXTRACT_PATH}")
        return LORA_EXTRACT_PATH

    # Then check subfolders
    for item in os.listdir(LORA_EXTRACT_PATH):
        sub_path = os.path.join(LORA_EXTRACT_PATH, item)
        if os.path.isdir(sub_path) and os.path.exists(os.path.join(sub_path, "adapter_config.json")):
            print(f"Adapter ready: {sub_path}")
            return sub_path

    print(f"Extraction completed, adapter config not detected in: {LORA_EXTRACT_PATH}")
    return LORA_EXTRACT_PATH

lora_result = find_and_extract_lora()
print(f"LoRA path for next cell: {lora_result}")

In [ ]:
# Cell 3: Load models (strict LoRA classification on Llama 3 8B)

import os
import re
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print("=" * 60)
print("LOADING MODELS")
print("=" * 60)

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory before load: {torch.cuda.memory_allocated()/1e9:.1f} GB")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    for secret_name in ["HF_TOKEN", "huggingface_token", "HUGGING_FACE_TOKEN", "LLAMA_API"]:
        try:
            HF_TOKEN = user_secrets.get_secret(secret_name)
            if HF_TOKEN:
                print(f"HF token loaded from: {secret_name}")
                break
        except:
            pass
except:
    pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get("LLAMA_API")

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)

CLASS_MODEL_ID = "meta-llama/Meta-Llama-3-8B"
GEN_MODEL_ID = CLASS_MODEL_ID

LOCAL_MODEL_CANDIDATES = [
    "/kaggle/input/datasets/ranjaysingh07/llama-8b-trained-model",
    "/kaggle/input/llama-8b-trained-model",
    "/kaggle/working/llama-8b-trained-model",
]

def find_local_model_path():
    for p in LOCAL_MODEL_CANDIDATES:
        if os.path.isdir(p) and os.path.exists(os.path.join(p, "config.json")):
            return p
    return None

resolved_model_source = find_local_model_path() or CLASS_MODEL_ID
if os.path.isdir(resolved_model_source):
    print(f"Classification model source: local path -> {resolved_model_source}")
else:
    print(f"Classification model source: hub repo -> {resolved_model_source}")
print(f"Generation fallback model: {GEN_MODEL_ID}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        resolved_model_source,
        token=HF_TOKEN,
        local_files_only=os.path.isdir(resolved_model_source),
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        resolved_model_source,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
        local_files_only=os.path.isdir(resolved_model_source),
    )
except Exception as e:
    if os.path.isdir(resolved_model_source):
        raise RuntimeError(
            f"Failed to load local Llama model from {resolved_model_source}. "
            "Ensure the folder contains full model files (config.json, tokenizer files, weights)."
        ) from e
    raise RuntimeError(
        "Failed to load gated Hub model meta-llama/Meta-Llama-3-8B. "
        "Accept the model license on Hugging Face and set a valid HF token in Kaggle secrets (HF_TOKEN), "
        "or provide a local model folder in Kaggle input with full model files."
    ) from e

tokenizer.pad_token = tokenizer.eos_token
print(f"Base model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

print("Loading LoRA adapter for strict classification...")
adapter_loaded = False
class_model = None

adapter_path = None
search_paths = [
    LORA_EXTRACT_PATH,
    f"{LORA_EXTRACT_PATH}/lora_adapter",
    "/kaggle/working/llama_lora",
    "/kaggle/working/llama_lora/lora_adapter",
    "/kaggle/input/datasets/ranjaysingh07/llama-8b-trained-model",
]

if os.path.exists("/kaggle/input"):
    for folder in os.listdir("/kaggle/input"):
        folder_path = f"/kaggle/input/{folder}"
        search_paths.append(folder_path)
        if os.path.isdir(folder_path):
            for sub in os.listdir(folder_path):
                search_paths.append(os.path.join(folder_path, sub))

for path in search_paths:
    if os.path.exists(os.path.join(path, "adapter_config.json")):
        adapter_path = path
        break

if not adapter_path:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            adapter_path = root
            break
    if not adapter_path:
        for root, dirs, files in os.walk("/kaggle/working"):
            if "adapter_config.json" in files:
                adapter_path = root
                break

if not adapter_path:
    raise RuntimeError(
        "Strict mode enabled: trained LoRA adapter is required for classification, but adapter_config.json was not found."
    )

print(f"Adapter found: {adapter_path}")
try:
    class_model = PeftModel.from_pretrained(base_model, adapter_path)
    adapter_loaded = True
    print("LoRA adapter loaded")
    print(f"GPU memory after adapter: {torch.cuda.memory_allocated()/1e9:.1f} GB")
except Exception as e:
    raise RuntimeError(
        f"Strict mode enabled: failed to load trained LoRA adapter from {adapter_path}."
    ) from e

gen_model = class_model
gen_tokenizer = tokenizer
class_tokenizer = tokenizer
MODEL_ID = CLASS_MODEL_ID
class_model.eval()

print("=" * 60)
print("MODEL READY")
print(f"Classification model: {CLASS_MODEL_ID}")
print("LoRA status: LOADED (strict mode)")
print(f"GPU memory now: {torch.cuda.memory_allocated()/1e9:.1f} GB")
print("=" * 60)

In [ ]:

# Cell 4: Define Classification Categories (My 7 Categories)

# My 7 trained classification categories
CATEGORIES = [
    "SIMPLE INSTRUCTION",
    "INSTRUCTION WITH SEQUENCE",
    "PARALLEL INSTRUCTION",
    "INSTRUCTION WITH PURPOSE",
    "INSTRUCTION WITH REASON",
    "EXCLUSIVE INSTRUCTION (OBJECTS)",
    "EXCLUSIVE INSTRUCTION (ACTIONS)"
]

# Classification instruction (matching my training format exactly)
CLASSIFICATION_INSTRUCTION = """Classify the instruction type into one of: SIMPLE INSTRUCTION, INSTRUCTION WITH SEQUENCE, PARALLEL INSTRUCTION, INSTRUCTION WITH PURPOSE, INSTRUCTION WITH REASON, EXCLUSIVE INSTRUCTION (OBJECTS), EXCLUSIVE INSTRUCTION (ACTIONS)."""

# Category descriptions
CATEGORY_INFO = {
    "SIMPLE INSTRUCTION": "A basic, single action instruction",
    "INSTRUCTION WITH SEQUENCE": "Steps in order (then, after, first, next)",
    "PARALLEL INSTRUCTION": "Multiple simultaneous actions (and, while, simultaneously)",
    "INSTRUCTION WITH PURPOSE": "Action with goal (to, for, in order to, so that)",
    "INSTRUCTION WITH REASON": "Action with explanation (because, since, as)",
    "EXCLUSIVE INSTRUCTION (OBJECTS)": "Choice between objects (use X or Y)",
    "EXCLUSIVE INSTRUCTION (ACTIONS)": "Choice between actions (do X or do Y)"
}

print("My 7 Classification Categories:")
print("="*60)
for i, (cat, desc) in enumerate(CATEGORY_INFO.items(), 1):
    print(f"  {i}. {cat}")
    print(f"     → {desc}")
print("="*60)

In [ ]:
# Cell 5: Strict classification with real confidence (Llama 3 8B + trained LoRA)

import torch
import torch.nn.functional as F

print("Verifying strict LoRA classifier...")
if not adapter_loaded:
    raise RuntimeError("Strict mode requires trained LoRA adapter, but adapter is not loaded.")
print("LoRA adapter is loaded and will be used for all classifications")

def _candidate_avg_logprob(prompt_text, candidate_text):
    """Average token log-probability for candidate_text conditioned on prompt_text."""
    device = class_model.device

    prompt_ids = class_tokenizer(
        prompt_text,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    cand_ids = class_tokenizer(
        candidate_text,
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"]

    if cand_ids.shape[1] == 0:
        return float("-inf")

    full_ids = torch.cat([prompt_ids, cand_ids], dim=1).to(device)
    prompt_len = prompt_ids.shape[1]
    cand_len = cand_ids.shape[1]

    with torch.no_grad():
        out = class_model(full_ids)
        logits = out.logits[:, :-1, :]
        targets = full_ids[:, 1:]
        log_probs = F.log_softmax(logits, dim=-1)

    start = prompt_len - 1
    end = start + cand_len

    step_log_probs = log_probs[:, start:end, :].gather(
        -1, targets[:, start:end].unsqueeze(-1)
    ).squeeze(-1)

    return float(step_log_probs.mean().item())

def classify_instruction_with_confidence(instruction, debug=False):
    """Classify using trained LoRA only and return calibrated confidence over 7 classes."""
    if not adapter_loaded:
        raise RuntimeError("Strict mode: LoRA adapter is not loaded")

    prompt = f"""### Instruction:
{CLASSIFICATION_INSTRUCTION}

### Input:
{instruction}

### Response:
"""

    scores = []
    for category in CATEGORIES:
        s = _candidate_avg_logprob(prompt, category)
        scores.append(s)

    score_tensor = torch.tensor(scores, dtype=torch.float32)
    probs = torch.softmax(score_tensor, dim=0)
    best_idx = int(torch.argmax(probs).item())

    result = {
        "category": CATEGORIES[best_idx],
        "model_confidence": float(probs[best_idx].item()),
        "all_scores": {
            CATEGORIES[i]: float(probs[i].item()) for i in range(len(CATEGORIES))
        }
    }

    if debug:
        print(f"\nDEBUG input: {instruction}")
        for c in CATEGORIES:
            print(f"  {c:<38} -> {result['all_scores'][c]:.4f}")
        print(f"  Predicted: {result['category']} ({result['model_confidence']:.4f})")

    return result

# Quick validation
print("=" * 60)
print("STRICT CLASSIFIER SANITY TEST")
print("=" * 60)

sample_cases = [
    "Boil water",
    "Pour water then add salt",
    "Use coffee or tea",
    "If it is cold, wear a jacket",
]

for s in sample_cases:
    pred = classify_instruction_with_confidence(s, debug=True)
    print(f"-> {s} => {pred['category']} ({pred['model_confidence']:.4f})")

In [ ]:
# Cell 6: Instruction generation (Gemini first, local Llama fallback)
import os
import re
import requests

def get_target_step_count(query, context_text=None, summary_mode=False):
    q = (query or '').lower()
    if summary_mode:
        return 8

    score = 10
    detail_terms = ["detailed", "complete", "comprehensive", "full", "step by step", "from scratch", "guide"]
    if any(t in q for t in detail_terms):
        score += 2

    domain_terms = ["install", "setup", "configure", "deploy", "recipe", "prepare", "application", "account", "gmail"]
    if any(t in q for t in domain_terms):
        score += 1

    token_count = len(re.findall(r"[a-zA-Z0-9]+", q))
    if token_count >= 8:
        score += 1

    if context_text:
        context_lines = [x.strip() for x in str(context_text).split("\n") if x.strip()]
        score += min(2, max(0, len(context_lines) // 8))

    return max(10, min(15, score))

def load_gemini_api_key():
    key_names = ["GEMINI_API_KEY", "GOOGLE_API_KEY", "GENAI_API_KEY"]
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        for name in key_names:
            try:
                value = user_secrets.get_secret(name)
                if value:
                    print(f"Gemini key loaded from: {name}")
                    return value
            except Exception:
                pass
    except Exception:
        pass

    for name in key_names:
        value = os.environ.get(name)
        if value:
            print(f"Gemini key loaded from env: {name}")
            return value

    print("Gemini key not found. Using local Llama fallback generation.")
    return None

GEMINI_API_KEY = load_gemini_api_key()
GEMINI_API_VERSIONS = ["v1beta", "v1"]
GEMINI_MODEL_PREFERENCES = [
    "gemini-2.5-flash-lite",
    "gemini-2.5-flash",
    "gemini-3-flash-lite",
    "gemini-3-flash",
    "gemini-2.0-flash-lite",
    "gemini-2.0-flash",
    "gemini-1.5-flash-8b",
    "gemini-1.5-flash",
    "gemini-1.5-pro",
]

ACTIVE_GEMINI_TARGET = None
DISCOVERED_MODELS_BY_VERSION = {}

def list_available_gemini_models(api_version):
    if not GEMINI_API_KEY:
        return []
    if api_version in DISCOVERED_MODELS_BY_VERSION:
        return DISCOVERED_MODELS_BY_VERSION[api_version]

    url = f"https://generativelanguage.googleapis.com/{api_version}/models?key={GEMINI_API_KEY}"
    models = []
    try:
        response = requests.get(url, timeout=40)
        if response.status_code != 200:
            DISCOVERED_MODELS_BY_VERSION[api_version] = []
            return []

        payload = response.json()
        for model_info in payload.get("models", []):
            methods = model_info.get("supportedGenerationMethods", []) or []
            if "generateContent" not in methods:
                continue
            name = model_info.get("name", "")
            if name.startswith("models/"):
                name = name.split("models/", 1)[1]
            if name.startswith("gemini") and name not in models:
                models.append(name)
    except Exception:
        models = []

    DISCOVERED_MODELS_BY_VERSION[api_version] = models
    return models

def build_model_targets():
    targets = []
    if ACTIVE_GEMINI_TARGET:
        targets.append(ACTIVE_GEMINI_TARGET)

    discovered_by_version = {
        version: list_available_gemini_models(version) for version in GEMINI_API_VERSIONS
    }

    for version in GEMINI_API_VERSIONS:
        available = discovered_by_version.get(version, [])
        for preferred in GEMINI_MODEL_PREFERENCES:
            if preferred in available:
                target = (version, preferred)
                if target not in targets:
                    targets.append(target)
        extra = [m for m in available if ("flash" in m.lower()) and m not in GEMINI_MODEL_PREFERENCES]
        for m in extra:
            target = (version, m)
            if target not in targets:
                targets.append(target)

    for version in GEMINI_API_VERSIONS:
        for preferred in GEMINI_MODEL_PREFERENCES:
            target = (version, preferred)
            if target not in targets:
                targets.append(target)

    return targets

def call_gemini(prompt, temperature=0.2, max_output_tokens=1200):
    global ACTIVE_GEMINI_TARGET
    if not GEMINI_API_KEY:
        return None

    targets = build_model_targets()
    last_error = None

    for api_version, model in targets:
        url = f"https://generativelanguage.googleapis.com/{api_version}/models/{model}:generateContent?key={GEMINI_API_KEY}"
        payload = {
            "contents": [{"parts": [{"text": prompt}]}],
            "generationConfig": {
                "temperature": temperature,
                "topP": 0.9,
                "topK": 40,
                "maxOutputTokens": max_output_tokens,
            },
        }

        try:
            response = requests.post(url, json=payload, timeout=90)
            if response.status_code == 200:
                data = response.json()
                candidates = data.get("candidates", [])
                if not candidates:
                    continue
                parts = candidates[0].get("content", {}).get("parts", [])
                text = "\n".join([p.get("text", "") for p in parts]).strip()
                if text:
                    if ACTIVE_GEMINI_TARGET != (api_version, model):
                        print(f"Gemini target selected: {model} ({api_version})")
                    ACTIVE_GEMINI_TARGET = (api_version, model)
                    return text
                continue
            if response.status_code == 404:
                continue
            last_error = f"{response.status_code} - {response.text[:300]}"
        except Exception as e:
            last_error = str(e)

    if last_error:
        print(f"Gemini API error after trying candidates: {last_error}")
    else:
        print("No compatible Gemini model available. Using local Llama fallback.")
    return None

def parse_numbered_steps(text):
    if not text:
        return []

    lines = [x.strip() for x in text.split('\n') if x.strip()]
    steps = []
    for line in lines:
        cleaned = re.sub(r'^(?:Step\s*)?\d+[\.\)\:\s-]+', '', line, flags=re.IGNORECASE).strip()
        cleaned = re.sub(r'^[\-\*]+\s*', '', cleaned).strip()
        cleaned = re.sub(r'\*+', '', cleaned).strip()
        cleaned = re.sub(r'\s+', ' ', cleaned)
        if cleaned and 3 <= len(cleaned) <= 140:
            steps.append(cleaned)

    if not steps:
        sentence_chunks = re.split(r'(?<=[.!?])\s+', re.sub(r'\s+', ' ', text))
        for s in sentence_chunks:
            s = s.strip(" -*\n\t")
            if 4 <= len(s) <= 140:
                steps.append(s)

    dedup = []
    seen = set()
    for s in steps:
        key = s.lower()[:130]
        if key not in seen:
            seen.add(key)
            dedup.append(s)
    return dedup

def compact_instruction(text):
    if not text:
        return ""
    t = re.sub(r'\s+', ' ', text).strip()
    t = re.sub(r'^[\-\*]+\s*', '', t)
    t = t.strip(" .;:,-")
    t = re.sub(r'\b(typically|generally|usually|carefully|recommended|desired)\b', '', t, flags=re.IGNORECASE)
    t = re.sub(r'\s+', ' ', t).strip(" .;:,-")
    words = t.split()
    if len(words) > 12:
        t = " ".join(words[:12]).strip(" .;:,-")
    if t:
        t = t[0].upper() + t[1:]
    return t

def postprocess_compact_steps(steps, max_steps):
    compacted = []
    seen = set()
    for raw in steps:
        short = compact_instruction(raw)
        if not short:
            continue
        key = short.lower()
        if key not in seen:
            seen.add(key)
            compacted.append(short)
        if len(compacted) >= max_steps:
            break
    return compacted

def build_gemini_prompt(query, target_steps, context_text=None, require_variety=False):
    context_block = ""
    if context_text:
        context_block = f"""
Website Content:
{context_text[:9000]}

STRICT WEBSITE RULES:
- Use ONLY website content facts.
- Keep wording faithful to website content.
- If info is missing, write: Information not found in website content.
"""

    variety_rules = ""
    if require_variety:
        variety_rules = """
Additional diversity rules:
- Use at least one line with 'then'.
- Use at least one line with 'and'.
- Use at least one line with 'or'.
- Use at least one line with 'if you want'.
- Use at least one line starting with 'If ...'.
"""

    return f"""You generate dataset-style instruction lines.

Task: Create exactly {target_steps} short instructions for this query.

Query:
{query}
{context_block}
Strict format rules:
1) Return only numbered lines (no heading, no explanation).
2) Keep each line short (3-12 words).
3) One direct action phrase per line.
4) Use simple plain language.
5) Do not output long paragraphs.
6) Use connectors where needed: then / and / or / if you want / If ...
{variety_rules}
Category style guide:
- SIMPLE: one direct action.
- PURPOSE: action + 'if you want ...'.
- EXCLUSIVE OBJECTS: same action + object A or object B.
- EXCLUSIVE ACTIONS: action A or action B.
- SEQUENCE: action A then action B.
- PARALLEL: action A and action B.
- REASON: If condition, action.

Output example:
1. Read the script carefully
2. Coordinate with director then brief actors
3. Prepare props and check lighting
"""

def generate_instructions_local_fallback(query, num_steps=8):
    prompt = f"""Task: Generate short dataset-style instructions.
Query: {query}
Return exactly {num_steps} numbered lines, each 3-12 words.
Use concise imperative style.
Steps:
1."""

    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        outputs = gen_model.generate(
            **inputs,
            max_new_tokens=450,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            top_k=40,
            repetition_penalty=1.1,
            pad_token_id=gen_tokenizer.eos_token_id,
        )

    response = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "Steps:" in response:
        response = response.split("Steps:")[-1]
    parsed = parse_numbered_steps(response)
    return postprocess_compact_steps(parsed, num_steps)

def generate_instructions(query, num_steps=None, context_text=None, require_variety=False, summary_mode=False):
    print(f"Generating instructions for: '{query}'")

    if num_steps is None:
        num_steps = get_target_step_count(query, context_text=context_text, summary_mode=summary_mode)

    prompt = build_gemini_prompt(
        query,
        target_steps=num_steps,
        context_text=context_text,
        require_variety=require_variety,
    )
    gemini_output = call_gemini(prompt)

    if gemini_output:
        steps = parse_numbered_steps(gemini_output)
        steps = postprocess_compact_steps(steps, num_steps)
        if steps:
            print(f"Extracted {len(steps)} Gemini steps")
            return steps

    print("Using local Llama fallback generation")
    fallback_steps = generate_instructions_local_fallback(query, num_steps=num_steps)
    if fallback_steps:
        print(f"Extracted {len(fallback_steps)} local fallback steps")
        return fallback_steps

    return ["Create a clear instruction"]

In [ ]:
# Cell 7: Processing Functions
def classify_independently(step_text):
    """Strict classifier wrapper: trained LoRA only, with real confidence."""
    pred = classify_instruction_with_confidence(step_text)
    return {
        "category": pred["category"],
        "source": "lora-model",
        "model_confidence": round(pred["model_confidence"], 4)
    }

def extend_steps_from_content(existing_steps, context_lines, target_steps):
    if len(existing_steps) >= target_steps:
        return existing_steps[:target_steps]

    expanded = list(existing_steps)
    seen = {s.lower().strip() for s in expanded if s}

    for line in context_lines:
        if len(expanded) >= target_steps:
            break
        clean = re.sub(r'\s+', ' ', line).strip(" .")
        if 4 <= len(clean) <= 120 and clean.lower() not in seen:
            expanded.append(clean)
            seen.add(clean.lower())

    if len(expanded) < target_steps:
        blob = " ".join(context_lines)
        chunks = re.split(r'(?<=[.!?])\s+', blob)
        for chunk in chunks:
            if len(expanded) >= target_steps:
                break
            clean = re.sub(r'\s+', ' ', chunk).strip(" .")
            if 5 <= len(clean) <= 120 and clean.lower() not in seen:
                expanded.append(clean)
                seen.add(clean.lower())

    return expanded[:target_steps]

def process_query(query):
    print("Mode: GENERATE from query")
    print(f"   Query: {query}")
    print("   Classifier: trained LoRA (strict)")

    steps = generate_instructions(query, require_variety=True)
    print(f"   Generated {len(steps)} steps")

    results = []
    for i, step in enumerate(steps, 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        }
        results.append(item)
        print(f"   {i}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {step[:80]}...")

    return {
        "mode": "generate",
        "query": query,
        "classifier": "lora-strict",
        "instructions": results
    }

def process_paragraph(paragraph):
    print("Mode: CLASSIFY paragraph")
    print(f"   Length: {len(paragraph)} chars")
    print("   Classifier: trained LoRA (strict)")

    sentences = re.split(r'(?<=[.!?])\s+', paragraph.strip())
    results = []
    step_num = 1
    valid_sentences = [re.sub(r'\s+', ' ', s.strip()) for s in sentences if len(s.strip()) > 5]

    for sentence in valid_sentences:
        meta = classify_independently(sentence)
        category = meta["category"]
        results.append({
            "step": step_num,
            "instruction": sentence,
            "category": category,
            "input": sentence,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        })
        print(f"   {step_num}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {sentence[:80]}...")
        step_num += 1

    return {
        "mode": "classify",
        "query": paragraph[:100] + "..." if len(paragraph) > 100 else paragraph,
        "classifier": "lora-strict",
        "instructions": results
    }

def is_summary_query(query):
    q = (query or "").lower()
    summary_terms = [
        "summarize", "summarise", "summerize", "summary", "overview", "key points",
        "main points", "what is this page", "website data", "web data",
        "give me my web", "extract data", "page data", "content from page"
    ]
    return any(term in q for term in summary_terms)

def query_tokens(query):
    q = re.sub(r'[^a-zA-Z0-9\s]', ' ', (query or '').lower())
    stop = {
        'the', 'is', 'a', 'an', 'to', 'for', 'and', 'or', 'of', 'on', 'in', 'with',
        'from', 'this', 'that', 'page', 'website', 'web', 'site', 'my', 'me', 'give',
        'according', 'based', 'please'
    }
    return [t for t in q.split() if len(t) > 2 and t not in stop]

def extract_ordered_instructions(content):
    lines = [re.sub(r'\s+', ' ', l).strip() for l in content.split('\n')]
    lines = [l for l in lines if l]

    numbered = []
    for line in lines:
        m = re.match(r'^\s*(\d{1,2})[\.)\-:]\s+(.*)$', line)
        if m:
            step_text = m.group(2).strip()
            if 6 <= len(step_text) <= 260:
                numbered.append(step_text)

    dedup = []
    seen = set()
    for x in numbered:
        k = x.lower()[:120]
        if k not in seen:
            seen.add(k)
            dedup.append(x)

    return dedup

def extract_page_snippets(content, max_items=12):
    if not content:
        return []

    lines = [re.sub(r'\s+', ' ', x).strip() for x in content.split('\n')]
    lines = [x for x in lines if 20 <= len(x) <= 280]

    noise_terms = [
        'http://', 'https://', 'www.', 'download article', 'last updated',
        'co-authored', 'fact checked', 'cookie', 'advertisement', 'subscribe',
        'comment', 'share this', 'privacy policy'
    ]
    lines = [x for x in lines if not any(t in x.lower() for t in noise_terms)]
    lines = [x for x in lines if not re.search(r'\b(thanks|thank you|hello|i live|can you help|dear)\b', x.lower())]

    if len(lines) < 4:
        sentence_chunks = re.split(r'(?<=[.!?])\s+', re.sub(r'\s+', ' ', content))
        lines.extend([x.strip() for x in sentence_chunks if 20 <= len(x.strip()) <= 280])

    deduped = []
    seen = set()
    for line in lines:
        key = line.lower()[:110]
        if key not in seen:
            seen.add(key)
            deduped.append(line)
        if len(deduped) >= max_items:
            break

    return deduped

def filter_by_query(snippets, query):
    tokens = query_tokens(query)
    if not tokens:
        return snippets

    scored = []
    for s in snippets:
        lower = s.lower()
        score = sum(1 for t in tokens if t in lower)
        if score > 0:
            scored.append((score, s))

    scored.sort(key=lambda x: x[0], reverse=True)
    if scored:
        return [x[1] for x in scored]
    return snippets

def process_website_content(content, query=None):
    print("Mode: WEBSITE content")
    print(f"   Content length: {len(content)} chars")
    print(f"   Query: {query if query else 'N/A'}")
    print("   Classifier: trained LoRA (strict)")

    ordered_steps = extract_ordered_instructions(content)
    snippets = extract_page_snippets(content, max_items=22)
    snippets = filter_by_query(snippets, query)

    q_tokens = query_tokens(query)
    page_blob = " ".join((ordered_steps[:25] + snippets[:25])).lower()
    overlap = sum(1 for t in q_tokens if t in page_blob)

    if q_tokens and overlap == 0:
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "Query topic not found in current webpage content. Disable page extraction or open a relevant page.",
            "classifier": "lora-strict",
            "instructions": []
        }

    if not snippets and not ordered_steps:
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "No usable content found on this page.",
            "classifier": "lora-strict",
            "instructions": []
        }

    context_lines = ordered_steps[:18] if ordered_steps else snippets[:18]
    context_text = "\n".join([f"- {x}" for x in context_lines])
    summary_mode = is_summary_query(query)
    if summary_mode:
        gemini_query = query or "Summarize this webpage professionally"
    else:
        gemini_query = query or "Extract complete professional instructions from this webpage"

    target_steps = get_target_step_count(gemini_query, context_text=context_text, summary_mode=summary_mode)
    structured_steps = generate_instructions(
        gemini_query,
        num_steps=target_steps,
        context_text=context_text,
        require_variety=False,
        summary_mode=summary_mode
    )
    structured_steps = extend_steps_from_content(structured_steps, context_lines, target_steps)

    results = []
    for i, step in enumerate(structured_steps[:15], 1):
        meta = classify_independently(step)
        category = meta["category"]
        item = {
            "step": i,
            "instruction": step,
            "category": category,
            "input": step,
            "output": category,
            "source": meta["source"],
            "model_confidence": meta["model_confidence"]
        }
        results.append(item)
        print(f"   {i}. [{category}] ({meta['source']} | conf={meta['model_confidence']}) {step[:80]}...")

    return {
        "mode": "website",
        "query": query or "Website Content Analysis",
        "summary": " ".join(snippets[:3]) if snippets else " ".join(structured_steps[:2]),
        "classifier": "lora-strict",
        "instructions": results
    }

def is_website_intent_query(query):
    q = (query or '').lower()
    website_terms = [
        'website', 'web site', 'webpage', 'web page', 'page content',
        'my page', 'this page', 'browser', 'chrome', 'site data', 'web data',
        'summarize my website', 'summarise my website', 'summerize my website', 'data of my webpage'
    ]
    return any(term in q for term in website_terms)

def process_request(data):
    query = data.get('query') or data.get('text')
    paragraph = data.get('paragraph')
    website_content = data.get('website_content') or data.get('pageContent')
    mode = data.get('mode', 'auto')

    if mode == 'auto':
        if paragraph:
            mode = 'classify'
        elif website_content or is_website_intent_query(query):
            mode = 'website'
        else:
            mode = 'generate'

    if mode == 'classify' and paragraph:
        return process_paragraph(paragraph)
    elif mode == 'website':
        if website_content:
            return process_website_content(website_content, query=query)
        return {
            "mode": "website",
            "query": query or "Website Content Analysis",
            "summary": "Website mode requested but no page content was received. Keep 'Extract relevant content from page' ON and reload the extension.",
            "classifier": "lora-strict",
            "instructions": []
        }
    elif query:
        return process_query(query)
    else:
        return {"error": "No valid input provided. Send query, paragraph, or website_content."}

print("Processing functions ready")
print("Classification uses strict trained LoRA with real confidence")

In [ ]:
# Cell 8: Configure ngrok using uploaded binary + uploaded yml
import os
import stat
import shutil
from pyngrok import ngrok, conf

NGROK_SOURCE_BIN_PATH = "/kaggle/input/datasets/ranjaysingh07/ngrok-file/ngrok"
NGROK_CONFIG_PATH = "/kaggle/input/datasets/ranjaysingh07/ngrok-yml/ngrok.yml"
NGROK_WORKING_BIN_PATH = "/kaggle/working/ngrok"

if not os.path.exists(NGROK_SOURCE_BIN_PATH):
    raise FileNotFoundError(f"ngrok binary not found: {NGROK_SOURCE_BIN_PATH}")
if not os.path.exists(NGROK_CONFIG_PATH):
    raise FileNotFoundError(f"ngrok config file not found: {NGROK_CONFIG_PATH}")

# /kaggle/input is read-only, so copy binary to writable /kaggle/working
shutil.copy2(NGROK_SOURCE_BIN_PATH, NGROK_WORKING_BIN_PATH)

# Ensure copied binary is executable
os.chmod(NGROK_WORKING_BIN_PATH, os.stat(NGROK_WORKING_BIN_PATH).st_mode | stat.S_IEXEC)

# Build pyngrok config from uploaded yml + writable binary (no download required)
PYNGROK_CONFIG = conf.PyngrokConfig(
    ngrok_path=NGROK_WORKING_BIN_PATH,
    config_path=NGROK_CONFIG_PATH
)

print("ngrok configured from uploaded files")
print(f"source binary: {NGROK_SOURCE_BIN_PATH}")
print(f"working binary: {NGROK_WORKING_BIN_PATH}")
print(f"config: {NGROK_CONFIG_PATH}")

In [ ]:
# Cell 9: Flask Server with All Endpoints

from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
import traceback

app = Flask(__name__)

# CORS Helper

def cors_response(data, status=200):
    """Create response with CORS headers"""
    response = jsonify(data)
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = 'Content-Type'
    response.headers['Access-Control-Allow-Methods'] = 'GET, POST, OPTIONS'
    return response, status


def handle_options():
    """Handle OPTIONS preflight requests"""
    return cors_response({'status': 'ok'})


# ENDPOINT 1: /parse - MAIN ENDPOINT (auto-detects mode)

@app.route('/parse', methods=['POST', 'OPTIONS'])
def parse():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        
        if not any([data.get('query'), data.get('text'), data.get('paragraph'), 
                    data.get('website_content'), data.get('pageContent')]):
            return cors_response({'error': 'No input provided'}, 400)
        
        result = process_request(data)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)


# ENDPOINT 2: /generate - Generate instructions from query

@app.route('/generate', methods=['POST', 'OPTIONS'])
def generate():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        query = data.get('query') or data.get('text', '')
        
        if not query:
            return cors_response({'error': 'No query provided'}, 400)
        
        result = process_query(query)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 3: /classify - Classify a paragraph

@app.route('/classify', methods=['POST', 'OPTIONS'])
def classify():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        paragraph = data.get('paragraph') or data.get('text', '')
        
        if not paragraph:
            return cors_response({'error': 'No paragraph provided'}, 400)
        
        result = process_paragraph(paragraph)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 4: /website - Process website content

@app.route('/website', methods=['POST', 'OPTIONS'])
def website():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        content = data.get('content') or data.get('website_content') or data.get('pageContent') or data.get('text', '')
        query = data.get('query')
        
        if not content:
            return cors_response({'error': 'No content provided'}, 400)
        
        result = process_website_content(content, query=query)
        return cors_response({'result': result})
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 5: /classify-single - Classify a single instruction

@app.route('/classify-single', methods=['POST', 'OPTIONS'])
def classify_single():
    if request.method == 'OPTIONS':
        return handle_options()
    
    try:
        data = request.json or {}
        instruction = data.get('instruction') or data.get('text', '')
        
        if not instruction:
            return cors_response({'error': 'No instruction provided'}, 400)
        
        meta = classify_independently(instruction)
        return cors_response({
            'result': {
                'instruction': instruction,
                'category': meta['category'],
                'source': meta['source'],
                'model_confidence': meta['model_confidence']
            }
        })
    
    except Exception as e:
        traceback.print_exc()
        return cors_response({'error': str(e)}, 500)

# ENDPOINT 6: /health - Health check

@app.route('/health', methods=['GET'])
def health():
    lora_status = "loaded" if adapter_loaded else "not loaded"
    return cors_response({
        'status': 'ok',
        'model': 'Llama 3 8B + optional LoRA',
        'classification_model': CLASS_MODEL_ID,
        'generation_fallback_model': GEN_MODEL_ID,
        'lora_status': lora_status,
        'categories': CATEGORIES,
        'endpoints': ['/parse', '/generate', '/classify', '/website', '/classify-single', '/health']
    })[0]


# START SERVER

print("\n" + "="*70)
print("STARTING INSTRUCTION STRUCTURER SERVER")
print("="*70)

# Use config prepared in Cell 8
pyngrok_config = globals().get("PYNGROK_CONFIG")
if pyngrok_config is None:
    # Fallback config (expects writable ngrok binary path)
    pyngrok_config = conf.PyngrokConfig(
        ngrok_path="/kaggle/working/ngrok",
        config_path="/kaggle/input/datasets/ranjaysingh07/ngrok-yml/ngrok.yml"
    )

# Start ngrok tunnel
public_url = ngrok.connect(5000, pyngrok_config=pyngrok_config)
url = public_url.public_url

lora_msg = "LoRA LOADED" if adapter_loaded else "NO LoRA"

print(f"""SERVER IS RUNNING!
My SERVER URL (copy for Chrome extension): {url:<55}
Classification model: {CLASS_MODEL_ID}
LoRA status: {lora_msg}
""")

# Run the Flask server (this blocks)
app.run(host="0.0.0.0", port=5000)